In [34]:
import pandas as pd

train = pd.read_csv('gas_train.csv')
test  = pd.read_csv('gas_test.csv')

print(train.shape, test.shape)

(3196, 6) (1476, 5)


In [35]:
train.head()

,시군구명,생활및판매,공공문화,복지의료,업무오락체육,총가스사용량
0,구로구,2,0,0,0,9077.8
1,구로구,6,0,1,2,10105.5
2,구로구,27,0,0,0,8603.6
3,구로구,2,0,0,0,11076.8
4,구로구,83,0,1,19,10781.4


In [36]:
test.head()

,시군구명,생활및판매,공공문화,복지의료,업무오락체육
0,구로구,2,0,0,0
1,구로구,6,0,1,2
2,구로구,27,0,0,0
3,구로구,2,0,0,0
4,구로구,80,0,1,19


In [37]:
train.isnull().sum()

시군구명      0
생활및판매     0
공공문화      0
복지의료      0
업무오락체육    0
총가스사용량    0
dtype: int64

In [38]:
test.isnull().sum()

시군구명      0
생활및판매     0
공공문화      0
복지의료      0
업무오락체육    0
dtype: int64

In [ ]:
target = train.pop('총가스사용량')
combined = pd.concat([train, test])
combined = pd.get_dummies(combined)

pvt = len(train)
train = combined[:pvt]
test  = combined[pvt:]

,생활및판매,공공문화,복지의료,업무오락체육,시군구명_강남구,시군구명_강동구,시군구명_강북구,시군구명_관악구,시군구명_광진구,시군구명_구로구,...,시군구명_서초구,시군구명_성동구,시군구명_성북구,시군구명_송파구,시군구명_양천구,시군구명_영등포구,시군구명_용산구,시군구명_은평구,시군구명_종로구,시군구명_중구
0,2,0,0,0,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,6,0,1,2,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2,27,0,0,0,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,2,0,0,0,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
4,83,0,1,19,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

(2556, 25) (640, 25)
(2556,) (640,)


In [42]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=0)
rf.fit(X_train, y_train)
pred_a = rf.predict(X_val)

from sklearn.metrics import root_mean_squared_error

rf_score = root_mean_squared_error(y_val, pred_a)
rf_score

961.5738115597597

In [ ]:
from lightgbm import LGBMRegressor

lg = LGBMRegressor(random_state=0)
lg.fit(X_train, y_train)
pred_b = lg.predict(X_val)

lg_score = root_mean_squared_error(y_val, pred_b)
lg_score

# RMSE가 낮을수록 더 좋은 모델

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000514 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 288
[LightGBM] [Info] Number of data points in the train set: 2556, number of used features: 25
[LightGBM] [Info] Start training from score 10352.757353


1064.8095758723994

In [45]:
pred = rf.predict(test)

pd.DataFrame({'pred': pred}).to_csv('result.csv', index=False)